In [15]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
import random
import copy
#Setting a random seed to ensure experiments are reproducible
seed=42
# 1. Python random
random.seed(seed)

# 2. NumPy
np.random.seed(seed)

# 3. PyTorch (CPU)
torch.manual_seed(seed)

# 4. PyTorch (GPU)
torch.cuda.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# 5. Deterministic behavior
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
'''
def createJobsTensorDataset(data_dir):
    #creates a Tensor from the Jobs Dataset given the directory of the .npz files
    npz =np.load(data_dir)
    X_tensor= torch.tensor(npz["x"][:,:,0], dtype=torch.float32) #takes the 0th slice of the replications
    y_tensor= torch.tensor(npz["yf"][:,0], dtype=torch.float32)
    t_tensor= torch.tensor(npz["t"][:,0], dtype=torch.float32)
    return TensorDataset(X_tensor,t_tensor,y_tensor)

jobs_train= createJobsTensorDataset("C:\\Users\\evany\\Desktop\\MaLabRotation\\Data\\FredJoData\\jobs_DW_bin.new.10.train.npz")
jobs_test= createJobsTensorDataset("C:\\Users\\evany\\Desktop\\MaLabRotation\\Data\\FredJoData\\jobs_DW_bin.new.10.test.npz")
'''
#train_loader = DataLoader(jobs_train, batch_size=64, shuffle=True)
#test_loader = DataLoader(jobs_test, batch_size=64, shuffle=False)

In [16]:
def createJobsTensorDataset(data_dir, split_by_e=True, return_type="both"):
    """
    Creates TensorDatasets from Jobs dataset using experimental indicator e.

    Args:
        data_dir (str): path to .npz file
        split_by_e (bool): whether to split dataset by e
        return_type (str): "both", "rct", or "obs"

    Returns:
        TensorDataset(s)
    """

    npz = np.load(data_dir)

    # Extract tensors (0th replication)
    X = torch.tensor(npz["x"][:, :, 0], dtype=torch.float32)
    y = torch.tensor(npz["yf"][:, 0], dtype=torch.float32)
    t = torch.tensor(npz["t"][:, 0], dtype=torch.float32)

    # Experimental indicator
    if "e" not in npz:
        raise ValueError("Dataset does not contain experimental indicator 'e'")

    e = torch.tensor(npz["e"][:, 0], dtype=torch.float32)

    if not split_by_e:
        return TensorDataset(X, t, y, e)

    # Masks
    rct_mask = (e == 1)
    obs_mask = (e == 0)

    # Split datasets
    rct_dataset = TensorDataset(
        X[rct_mask], t[rct_mask], y[rct_mask]
    )

    obs_dataset = TensorDataset(
        X[obs_mask], t[obs_mask], y[obs_mask]
    )

    # Return based on preference
    if return_type == "both":
        return rct_dataset, obs_dataset
    elif return_type == "rct":
        return rct_dataset
    elif return_type == "obs":
        return obs_dataset
    else:
        raise ValueError("return_type must be 'both', 'rct', or 'obs'")

In [17]:
jobs_train_rct, jobs_train_obs = createJobsTensorDataset(
    "C:\\Users\\evany\\Desktop\\MaLabRotation\\Data\\FredJoData\\jobs_DW_bin.new.10.train.npz",
    split_by_e=True,
    return_type="both"
)

jobs_test_rct, jobs_test_obs = createJobsTensorDataset(
    "C:\\Users\\evany\\Desktop\\MaLabRotation\\Data\\FredJoData\\jobs_DW_bin.new.10.test.npz",
    split_by_e=True,
    return_type="both"
)

In [18]:
#Exploring the covariates X of the Jobs Dataset
def explore_tensor(X):
    print("Shape:", X.shape)
    print("Mean:", X.mean(dim=0))
    print("Std:", X.std(dim=0))
    print("Min:", X.min(dim=0).values)
    print("Max:", X.max(dim=0).values)

    import pandas as pd
    import matplotlib.pyplot as plt
    
    df = pd.DataFrame(X.numpy())
    df.hist(figsize=(10,6), bins=20)
    plt.show()

#X_tensor = jobs_train.tensors[0]
#explore_tensor(X_tensor)

In [19]:
#Y_tensor = jobs_train.tensors[2]
#explore_tensor(Y_tensor)

In [20]:
#AutoEncoder: an Encoder paired with a Decoder to ensure x => z => x_recon is close
#Encoder: multi-layer feedforward network 
#Turns covariates X => representation Z

class AutoEncoder(nn.Module):
    def __init__(self,input_dim,hidden_dim,latent_dim):#,binary_index,cont_index
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim), # Hidden layer 1
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim//2), #Hidden Layer 2
            nn.ReLU(),
            nn.Linear(hidden_dim//2,latent_dim)
        )
        '''
        Stage 1
        Contains 3 output heads
        1. Reconstruction head using a decoder
        2. 2 Pseudooutcome heads trained on OBS data
        3. Propenssity head
        '''
        #Output heads
        #Reconstruction head via decoder
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim//2), # Hidden layer 1
            nn.ReLU(),
            nn.Linear(hidden_dim//2, hidden_dim), #Hidden Layer 2
            nn.ReLU(),
            nn.Linear(hidden_dim,input_dim)
        )


        #prospensity_head
        self.propensity_head= nn.Linear(latent_dim,1)

        #pseudo-outcome head, T=0, predicts a biased outcome estimation Y0
        self.t0_head =nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,1)
        )
        #pseudo-outcome head, T=1, predicts a biased outcome estimation Y1
        self.t1_head=nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim,1)
        )
        '''
        Stage 2
        1. Unconfounded outcome heads trained on RCT data
        '''
        self.g0_head = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )

        self.g1_head = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )


        
    def forward(self,x):
        z = self.encoder(x) #representation Z

        
        t_logit = self.propensity_head(z)
        x_recon = self.decoder(z)

        #pseudooutcomes
        y0_pseudo=self.t0_head(z)
        y1_pseudo=self.t1_head(z)

        # Stage 2 unconfounded outcomes
        y0_rct = self.g0_head(z)
        y1_rct = self.g1_head(z)
        return {
            "x_recon" :x_recon,
            "t_logit" :t_logit,
            "y0_pseudo":y0_pseudo,
            "y1_pseudo":y1_pseudo,
            "y0_rct": y0_rct,
            "y1_rct":y1_rct
        }





In [21]:
def pseudo_outcome_loss(y0_hat, y1_hat, t_batch, y_batch, outcome_type="continuous"):
    """
    Trains only the factual branch:
      - if t=0, compare y0_hat to y
      - if t=1, compare y1_hat to y
    """

    t_batch = t_batch.float()
    y_batch = y_batch.float()


    if outcome_type == "continuous":
        loss_fn = nn.MSELoss(reduction="none")
    elif outcome_type == "binary":
        loss_fn = nn.BCEWithLogitsLoss(reduction="none")
    else:
        raise ValueError("outcome_type must be 'continuous' or 'binary'")

    loss0 = loss_fn(y0_hat, y_batch)   # [B, 1]
    loss1 = loss_fn(y1_hat, y_batch)   # [B, 1]

    masked_loss = (1 - t_batch) * loss0 + t_batch * loss1
    return masked_loss.mean()

In [22]:
#Training Functions
def detect_binary_continuous_columns(X_tensor, tol=1e-6):
    """
    Detect binary vs continuous columns automatically.

    A column is considered binary if its unique values are only {0,1}
    within numerical tolerance

    Returns index of continuous and binary columns
    """
    binary_idx = []
    continuous_idx = []

    n_features = X_tensor.shape[1]

    for i in range(n_features):
        col = X_tensor[:, i]
        unique_vals = torch.unique(col)

        # Round tiny numerical noise
        unique_vals = torch.round(unique_vals / tol) * tol

        # Check whether all unique values are 0 or 1
        is_binary = torch.all((unique_vals == 0) | (unique_vals == 1)).item()

        if is_binary:
            binary_idx.append(i)
        else:
            continuous_idx.append(i)

    return binary_idx, continuous_idx

def train_mixed_autoencoder(
    dataset,
    hidden_dim=8,
    latent_dim=4,
    batch_size=64,
    lr=1e-3,
    num_epochs=50,
    verbose=True,
    device=None
):
    """
    Train an autoencoder on a TensorDataset(X, t, y) or TensorDataset(X).

    Automatically detects binary and continuous columns in X and uses:
      - BCEWithLogitsLoss for binary columns
      - MSELoss for continuous columns

    Returns:
      model, history, binary_idx, continuous_idx
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    # Extract X_tensor from TensorDataset
    if not hasattr(dataset, "tensors"):
        raise ValueError("dataset must be a TensorDataset")

    X_tensor = dataset.tensors[0].float()

    # Detect column types
    binary_idx, continuous_idx = detect_binary_continuous_columns(X_tensor)

    if verbose:
        print("Detected binary columns:", binary_idx)
        print("Detected continuous columns:", continuous_idx)

    input_dim = X_tensor.shape[1]
    model = AutoEncoder(input_dim=input_dim, hidden_dim=hidden_dim, latent_dim=latent_dim).to(device)

    mse = nn.MSELoss()
    bce = nn.BCEWithLogitsLoss()
    prop= nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    history = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_cont_loss = 0.0
        epoch_bin_loss = 0.0
        epoch_prop_loss =0.0
        epoch_pseudo_loss =0.0

        for X_batch,T_batch,Y_batch in loader:
            X_batch = X_batch.float().to(device)
            T_batch = T_batch.float().to(device)
            Y_batch = Y_batch.float().to(device)
            outputs = model(X_batch)
            #print("X_batch:", X_batch.shape)
            #print("Y_batch:", Y_batch.shape)
            #print("y0_hat:", outputs["y0_pseudo"].shape)
            #print("y1_hat:", outputs["y1_pseudo"].shape)
            #print("binary_idx:", binary_idx)
            #print("continuous_idx:", continuous_idx)
            
            
            loss = 0.0
            cont_loss = torch.tensor(0.0, device=device)
            bin_loss = torch.tensor(0.0, device=device)

            if len(continuous_idx) > 0:
                cont_pred = outputs["x_recon"][:, continuous_idx]
                cont_true = X_batch[:, continuous_idx]
                cont_loss = mse(cont_pred, cont_true)
                loss = loss + cont_loss

            if len(binary_idx) > 0:
                bin_pred = outputs["x_recon"][:, binary_idx]   # logits
                bin_true = X_batch[:, binary_idx]
                bin_loss = bce(bin_pred, bin_true)
                loss = loss + bin_loss

            #Propensity training
            T_batch = T_batch.unsqueeze(1).float()
            propensity_loss= prop(outputs["t_logit"],T_batch)
            loss+=propensity_loss

            #pseudooutcome training
            Y_batch = Y_batch.float().view(-1, 1)
            pseudo_loss = pseudo_outcome_loss(
                y0_hat = outputs["y0_pseudo"],
                y1_hat = outputs["y1_pseudo"],
                t_batch=T_batch,
                y_batch=Y_batch,
                outcome_type="binary"
            )
            loss +=pseudo_loss
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_cont_loss += cont_loss.item()
            epoch_bin_loss += bin_loss.item()
            epoch_prop_loss += propensity_loss.item()
            epoch_pseudo_loss += pseudo_loss.item()

        epoch_loss /= len(loader)
        epoch_cont_loss /= len(loader)
        epoch_bin_loss /= len(loader)
        epoch_prop_loss /= len(loader)
        epoch_pseudo_loss /= len(loader)

        history.append({
            "epoch": epoch + 1,
            "total_loss": epoch_loss,
            "continuous_loss": epoch_cont_loss,
            "binary_loss": epoch_bin_loss,
            "propensity_loss":epoch_prop_loss,
            "pseudo_outcome_loss":epoch_pseudo_loss
        })

        if verbose:
            print(
                f"Epoch {epoch+1}/{num_epochs} | "
                f"Total: {epoch_loss:.4f} | "
                f"Cont: {epoch_cont_loss:.4f} | "
                f"Bin: {epoch_bin_loss:.4f} | "
                f"Propensity: {epoch_prop_loss:.4f}|"
                f"Pseudo-outcome: {epoch_pseudo_loss:.4f}"
            )

    return model, history, binary_idx, continuous_idx

In [23]:
AutoEncoderModel, EncoderTrainHistory, binaryJobs, continuousJobs= train_mixed_autoencoder(jobs_train_obs)

Detected binary columns: [2, 3, 4, 5, 13, 14, 16]
Detected continuous columns: [0, 1, 6, 7, 8, 9, 10, 11, 12, 15]
Epoch 1/50 | Total: 3.5083 | Cont: 1.1958 | Bin: 0.7216 | Propensity: 0.7477|Pseudo-outcome: 0.8431
Epoch 2/50 | Total: 3.3823 | Cont: 1.1854 | Bin: 0.7007 | Propensity: 0.6945|Pseudo-outcome: 0.8017
Epoch 3/50 | Total: 3.2184 | Cont: 1.1543 | Bin: 0.6817 | Propensity: 0.6279|Pseudo-outcome: 0.7545
Epoch 4/50 | Total: 3.0427 | Cont: 1.1659 | Bin: 0.6616 | Propensity: 0.5244|Pseudo-outcome: 0.6908
Epoch 5/50 | Total: 2.7236 | Cont: 1.1109 | Bin: 0.6347 | Propensity: 0.3794|Pseudo-outcome: 0.5986
Epoch 6/50 | Total: 2.3297 | Cont: 1.0384 | Bin: 0.5830 | Propensity: 0.2227|Pseudo-outcome: 0.4856
Epoch 7/50 | Total: 1.9239 | Cont: 0.9342 | Bin: 0.4955 | Propensity: 0.1129|Pseudo-outcome: 0.3813
Epoch 8/50 | Total: 1.6520 | Cont: 0.8241 | Bin: 0.4294 | Propensity: 0.0686|Pseudo-outcome: 0.3299
Epoch 9/50 | Total: 1.4970 | Cont: 0.7304 | Bin: 0.3952 | Propensity: 0.0567|Pseudo-ou

In [26]:
#Stage 2 training
def initialize_stage2_from_stage1(model):
    model.g0_head.load_state_dict(copy.deepcopy(model.t0_head.state_dict()))
    model.g1_head.load_state_dict(copy.deepcopy(model.t1_head.state_dict()))

def freeze_encoder(model):
    for p in model.encoder.parameters():
        p.requires_grad = False

def unfreeze_encoder(model):
    for p in model.encoder.parameters():
        p.requires_grad = True

def freeze_module(module):
    for p in module.parameters():
        p.requires_grad = False


def unfreeze_module(module):
    for p in module.parameters():
        p.requires_grad = True

def clone_params(module):
    return {name: p.detach().clone() for name, p in module.named_parameters()}

def parameter_shift_loss(module, init_params):
    loss = 0.0
    for name, p in module.named_parameters():
        loss = loss + torch.sum((p - init_params[name]) ** 2)
    return loss

def train_stage2_rct(
    model,
    dataset,
    batch_size=64,
    lr=1e-3,
    num_epochs=50,
    lambda_shift=1e-4,
    outcome_type="binary",
    freeze_encoder=True,
    verbose=True,
    device=None
):
    """
    Train Stage 2 unconfounded outcome heads on RCT data only.

    Expects dataset = TensorDataset(X, T, Y)

    Trains:
      - y0_rct head on factual samples with T=0
      - y1_rct head on factual samples with T=1

    Optionally applies shift regularization so Stage 2 heads
    do not drift too far from their warm-start initialization.

    Returns:
      model, history
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    if not hasattr(dataset, "tensors"):
        raise ValueError("dataset must be a TensorDataset")

    if len(dataset.tensors) < 3:
        raise ValueError("dataset must be TensorDataset(X, T, Y)")

    model = model.to(device)

    # ---------------------------------
    # Freeze / unfreeze modules
    # ---------------------------------
    if freeze_encoder:
        freeze_module(model.encoder)
    else:
        unfreeze_module(model.encoder)

    # Freeze Stage 1 pieces
    freeze_module(model.decoder)
    freeze_module(model.propensity_head)
    freeze_module(model.t0_head)
    freeze_module(model.t1_head)

    # Unfreeze Stage 2 heads
    unfreeze_module(model.g0_head)
    unfreeze_module(model.g1_head)

    # ---------------------------------
    # Save initial params for shift loss
    # ---------------------------------
    g0_init = clone_params(model.g0_head)
    g1_init = clone_params(model.g1_head)

    # ---------------------------------
    # Optimizer only over trainable params
    # ---------------------------------
    optimizer = torch.optim.Adam(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=lr
    )

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    history = []

    for epoch in range(num_epochs):
        model.train()
        epoch_loss = 0.0
        epoch_rct_loss = 0.0
        epoch_shift_loss = 0.0

        for X_batch, T_batch, Y_batch in loader:
            X_batch = X_batch.float().to(device)
            T_batch = T_batch.float().to(device).view(-1, 1)
            Y_batch = Y_batch.float().to(device).view(-1, 1)

            outputs = model(X_batch)

            # Stage 2 factual loss using RCT heads
            rct_loss = pseudo_outcome_loss(
                y0_hat=outputs["y0_rct"],
                y1_hat=outputs["y1_rct"],
                t_batch=T_batch,
                y_batch=Y_batch,
                outcome_type=outcome_type
            )

            # shift penalty
            shift_loss = (
                parameter_shift_loss(model.g0_head, g0_init) +
                parameter_shift_loss(model.g1_head, g1_init)
            )

            loss = rct_loss + lambda_shift * shift_loss

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            epoch_loss += loss.item()
            epoch_rct_loss += rct_loss.item()
            epoch_shift_loss += shift_loss.item()

        epoch_loss /= len(loader)
        epoch_rct_loss /= len(loader)
        epoch_shift_loss /= len(loader)

        history.append({
            "epoch": epoch + 1,
            "total_loss": epoch_loss,
            "rct_outcome_loss": epoch_rct_loss,
            "shift_loss": epoch_shift_loss
        })

        if verbose:
            print(
                f"Epoch {epoch+1}/{num_epochs} | "
                f"Total: {epoch_loss:.4f} | "
                f"RCT outcome: {epoch_rct_loss:.4f} | "
                f"Shift: {epoch_shift_loss:.4f}"
            )

    return model, history
@torch.no_grad()
def predict_cate_no_ot(model, x, device="cpu"):
    model.eval()
    x = x.float().to(device)

    z = model.encode(x)

    cate_obs = model.t1_head(z) - model.t0_head(z)   # biased fallback
    cate_rct = model.g1_head(z) - model.g0_head(z)   # clean if z is on RCT manifold

    return {
        "cate_obs": cate_obs,
        "cate_rct": cate_rct
    }

@torch.no_grad()
def predict_cate_with_projection(model, z_obs, z_tilde, c):
    """
    z_obs:   original OBS latent point
    z_tilde: OT-barycentric projection onto RCT manifold
    c:       confidence score in [0,1], shape (batch,1)
    """
    cate_projected = model.g1_head(z_tilde) - model.g0_head(z_tilde)
    cate_fallback  = model.t1_head(z_obs)   - model.t0_head(z_obs)

    cate = c * cate_projected + (1.0 - c) * cate_fallback

    return {
        "cate_projected": cate_projected,
        "cate_fallback": cate_fallback,
        "cate": cate
    }

In [27]:
#Takes pseudo-outcome head parameters and uses it to inititialize unconfounded outcome heads
initialize_stage2_from_stage1(AutoEncoderModel)

AutoEncoderModel, stage2_history = train_stage2_rct(
    model=AutoEncoderModel,
    dataset=jobs_train_rct,
    batch_size=64,
    lr=1e-3,
    num_epochs=50,
    lambda_shift=1e-4,
    outcome_type="binary",
    freeze_encoder=True,
    verbose=True
)


Epoch 1/50 | Total: 0.7138 | RCT outcome: 0.7138 | Shift: 0.0013
Epoch 2/50 | Total: 0.7909 | RCT outcome: 0.7909 | Shift: 0.0071
Epoch 3/50 | Total: 0.7572 | RCT outcome: 0.7572 | Shift: 0.0178
Epoch 4/50 | Total: 0.6890 | RCT outcome: 0.6890 | Shift: 0.0341
Epoch 5/50 | Total: 0.7732 | RCT outcome: 0.7732 | Shift: 0.0530
Epoch 6/50 | Total: 0.6749 | RCT outcome: 0.6749 | Shift: 0.0749
Epoch 7/50 | Total: 0.6791 | RCT outcome: 0.6791 | Shift: 0.1000
Epoch 8/50 | Total: 0.6512 | RCT outcome: 0.6512 | Shift: 0.1279
Epoch 9/50 | Total: 0.6705 | RCT outcome: 0.6705 | Shift: 0.1577
Epoch 10/50 | Total: 0.6564 | RCT outcome: 0.6563 | Shift: 0.1901
Epoch 11/50 | Total: 0.6374 | RCT outcome: 0.6374 | Shift: 0.2228
Epoch 12/50 | Total: 0.6759 | RCT outcome: 0.6759 | Shift: 0.2547
Epoch 13/50 | Total: 0.6253 | RCT outcome: 0.6253 | Shift: 0.2874
Epoch 14/50 | Total: 0.6759 | RCT outcome: 0.6759 | Shift: 0.3216
Epoch 15/50 | Total: 0.6230 | RCT outcome: 0.6229 | Shift: 0.3582
Epoch 16/50 | Total

In [30]:
#Testing Function
def test_mixed_autoencoder(
    model,
    dataset,
    binary_idx=None,
    continuous_idx=None,
    batch_size=64,
    device=None,
    verbose=True
):
    """
    Test an autoencoder on a mixed dataset with binary and continuous features.

    Parameters
    ----------
    model : torch.nn.Module
        Model returning X_recon = model(X_batch)
    dataset : TensorDataset
        Usually TensorDataset(X, t, y) or TensorDataset(X)
    binary_idx : list or None
        Indices of binary columns. If None, detect automatically.
    continuous_idx : list or None
        Indices of continuous columns. If None, detect automatically.
    batch_size : int
        Batch size for evaluation
    device : str or None
        "cuda" or "cpu"
    verbose : bool
        Whether to print metrics

    Returns
    -------
    metrics : dict
        Dictionary with total loss, binary loss, continuous loss,
        continuous RMSE, and binary accuracy
    """

    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    model = model.to(device)
    model.eval()

    if not hasattr(dataset, "tensors"):
        raise ValueError("dataset must be a TensorDataset")

    X_tensor = dataset.tensors[0].float()

    if binary_idx is None or continuous_idx is None:
        binary_idx, continuous_idx = detect_binary_continuous_columns(X_tensor)

    loader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    mse = nn.MSELoss(reduction="sum")
    bce = nn.BCEWithLogitsLoss(reduction="sum")

    total_loss = 0.0
    total_cont_loss = 0.0
    total_bin_loss = 0.0
    total_prop_loss=0.0
    total_pseudo_loss=0.0

    total_cont_count = 0
    total_bin_count = 0
    correct_bin = 0
    correct_prop=0
    correct_pseudo=0
    total_prop_count=0
    total_pseudo_count=0

    with torch.no_grad():
        for X_batch, T_batch, Y_batch in loader:
            X_batch = X_batch.float().to(device)
            T_batch = T_batch.float().to(device)
            Y_batch = Y_batch.float().to(device)
            
            outputs = model(X_batch)
            X_recon = outputs["x_recon"]
            
            batch_loss = 0.0

            # Continuous part
            if len(continuous_idx) > 0:
                cont_true = X_batch[:, continuous_idx]
                cont_pred = X_recon[:, continuous_idx]

                cont_loss = mse(cont_pred, cont_true)
                total_cont_loss += cont_loss.item()
                total_cont_count += cont_true.numel()
                batch_loss += cont_loss

            # Binary part
            if len(binary_idx) > 0:
                bin_true = X_batch[:, binary_idx]
                bin_logits = X_recon[:, binary_idx]

                bin_loss = bce(bin_logits, bin_true)
                total_bin_loss += bin_loss.item()
                total_bin_count += bin_true.numel()
                batch_loss += bin_loss

                bin_probs = torch.sigmoid(bin_logits)
                bin_pred = (bin_probs > 0.5).float()
                correct_bin += (bin_pred == bin_true).sum().item()
            #Propensity Score Part
            t_true = T_batch.float().view(-1, 1)   # shape [batch, 1]
            t_logits = outputs["t_logit"]          # shape [batch, 1]

            prop_loss = bce(t_logits, t_true)
            total_prop_loss += prop_loss.item()
            total_prop_count += t_true.numel()
            batch_loss += prop_loss

            t_probs = torch.sigmoid(t_logits)
            t_pred = (t_probs > 0.5).float()
            correct_prop += (t_pred == t_true).sum().item()

            #Pseudo-outcome (binary Y)

            y_true = Y_batch.float().view(-1, 1)   # [B,1]

            y0_logits = outputs["y0_pseudo"]       # [B,1]
            y1_logits = outputs["y1_pseudo"]       # [B,1]

            # Compute losses per branch
            bce_none=nn.BCEWithLogitsLoss(reduction="none")
            loss0 = bce_none(y0_logits, y_true)         # [B,1] if reduction="none"
            loss1 = bce_none(y1_logits, y_true)

            # Masked factual loss
            pseudo_loss = (1 - t_true) * loss0 + t_true * loss1

            # Aggregate
            total_pseudo_loss += pseudo_loss.sum().item()
            total_pseudo_count += y_true.numel()
            batch_loss += pseudo_loss.mean()

            # Predictions (only factual branch matters)
            y0_probs = torch.sigmoid(y0_logits)
            y1_probs = torch.sigmoid(y1_logits)

            y_pred = (1 - t_true) * (y0_probs > 0.5).float() + t_true * (y1_probs > 0.5).float()

            correct_pseudo += (y_pred == y_true).sum().item()
            
            
            total_loss += batch_loss.item()

    # Normalize to per-element average
    avg_cont_loss = total_cont_loss / total_cont_count if total_cont_count > 0 else 0.0
    avg_bin_loss = total_bin_loss / total_bin_count if total_bin_count > 0 else 0.0
    avg_prop_loss = total_prop_loss / total_prop_count if total_prop_count > 0 else 0.0
    avg_pseudo_loss = total_pseudo_loss / total_pseudo_count if total_pseudo_count > 0 else 0.0
    avg_total_loss = 0.0

    denom = total_cont_count + total_bin_count + total_prop_count + total_pseudo_count
    if denom > 0:
        avg_total_loss = (total_cont_loss + total_bin_loss + total_prop_loss + total_pseudo_loss) / denom

    cont_rmse = avg_cont_loss ** 0.5 if total_cont_count > 0 else 0.0
    bin_accuracy = correct_bin / total_bin_count if total_bin_count > 0 else 0.0
    prop_accuracy = correct_prop / total_prop_count if total_prop_count > 0 else 0.0
    pseudo_accuracy = correct_pseudo / total_pseudo_count if total_pseudo_count > 0 else 0.0

    metrics = {
        "total_loss": avg_total_loss,
        "continuous_loss_mse": avg_cont_loss,
        "continuous_rmse": cont_rmse,
        "binary_loss_bce": avg_bin_loss,
        "binary_accuracy": bin_accuracy,
        "binary_idx": binary_idx,
        "continuous_idx": continuous_idx,
        "prop_loss_bce": avg_prop_loss,
        "prop_accuracy":prop_accuracy,
        "pseudo_loss":avg_pseudo_loss,
        "pseudo_accuracy":pseudo_accuracy
    }

    if verbose:
        print("Test results")
        print(f"Total loss:          {metrics['total_loss']:.4f}")
        print(f"Continuous MSE:      {metrics['continuous_loss_mse']:.4f}")
        print(f"Continuous RMSE:     {metrics['continuous_rmse']:.4f}")
        print(f"Binary BCE:          {metrics['binary_loss_bce']:.4f}")
        print(f"Binary accuracy:     {metrics['binary_accuracy']:.4f}")
        print(f"Propensity BCE Loss:     {metrics['prop_loss_bce']:.4f}")
        print(f"Propensity accuracy:     {metrics['prop_accuracy']:.4f}")
        print(f"Pseudo-outcome Binary Loss:     {metrics['pseudo_loss']:.4f}")
        print(f"Pseudo-outcome accuracy:     {metrics['pseudo_accuracy']:.4f}")
        print(f"Binary columns:      {metrics['binary_idx']}")
        print(f"Continuous columns:  {metrics['continuous_idx']}")

    return metrics

In [31]:
test_mixed_autoencoder(AutoEncoderModel, jobs_test)

Test results
Total loss:          0.2375
Continuous MSE:      0.2032
Continuous RMSE:     0.4508
Binary BCE:          0.2735
Binary accuracy:     0.8939
Propensity BCE Loss:     0.2066
Propensity accuracy:     0.9065
Pseudo-outcome Binary Loss:     0.3588
Pseudo-outcome accuracy:     0.8723
Binary columns:      [2, 3, 4, 5, 13, 14, 16]
Continuous columns:  [0, 1, 6, 7, 8, 9, 10, 11, 12, 15]


{'total_loss': 0.2374651693998741,
 'continuous_loss_mse': 0.2032243451596792,
 'continuous_rmse': 0.4508041095195109,
 'binary_loss_bce': 0.27345986572116227,
 'binary_accuracy': 0.8938584779706275,
 'binary_idx': [2, 3, 4, 5, 13, 14, 16],
 'continuous_idx': [0, 1, 6, 7, 8, 9, 10, 11, 12, 15],
 'prop_loss_bce': 0.20662070193394694,
 'prop_accuracy': 0.9065420560747663,
 'pseudo_loss': 0.35875500501873336,
 'pseudo_accuracy': 0.8722741433021807}